# LeetCode 298 — Binary Tree Longest Consecutive Sequence bằng Python

Link đề: https://leetcode.com/problems/binary-tree-longest-consecutive-sequence/ · Độ khó: **Medium** (Premium) · Chủ đề: Binary Tree, DFS, BFS, Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (với mỗi nút làm điểm cuối, đi ngược lên cha khi giá trị giảm đúng `1`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **độ dài** (số nút) của **đường đi liên tiếp tăng dần** dài nhất trong cây nhị phân, trong đó **mỗi nút lớn hơn nút trước đúng `1`** và đường đi **chỉ đi xuống** (từ cha tới con), **bắt đầu ở nút bất kỳ**.
2. Phân biệt **đường đi một chiều từ trên xuống** với **đường đi bất kỳ** (có thể đi lên cha rồi xuống) — bài này **không** cho đi từ con lên cha.
3. Đi từ cách **với mỗi nút, DFS xuống tìm đoạn liên tiếp dài nhất bắt đầu từ nó** `O(n²)` → **DFS từ trên xuống mang theo (giá trị cha, độ dài hiện tại)** `O(n)` → **DFS từ dưới lên trả về độ dài lớn nhất bắt đầu tại nút** → **lặp bằng stack / BFS** (tránh tràn đệ quy khi cây rất sâu).
4. Nắm kỹ thuật **truyền thông tin xuống** (top-down, tham số) so với **trả thông tin lên** (bottom-up, giá trị trả về), và cách giữ **biến toàn cục** `best`.
5. Tránh các lỗi kinh điển: cho phép **giảm dần** (`child.val == parent.val - 1`), **không reset** độ dài về `1` khi chuỗi đứt, so sánh với **gốc** thay vì **cha**, quên **cập nhật `best`** ở mọi nút, đếm **cạnh** thay vì **nút**, và **tràn đệ quy** ở cây lệch `3·10^4` nút.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân, trả về **độ dài của đường đi dãy liên tiếp (consecutive sequence path) dài nhất**.

**Đường đi dãy liên tiếp** là đường đi mà các giá trị **tăng thêm đúng `1`** dọc theo đường. Đường đi có thể **bắt đầu ở nút bất kỳ** trong cây, và **không thể** đi từ một nút lên nút cha.

**Ví dụ 1**

- Input: `root = [1,null,3,2,4,null,null,null,5]`
- Output: `3`
- Giải thích: đường đi dãy liên tiếp dài nhất là `3-4-5`, độ dài `3`.

**Ví dụ 2**

- Input: `root = [2,null,3,2,null,1]`
- Output: `2`
- Giải thích: đường đi dài nhất là `2-3` (độ dài `2`), **không phải** `3-2-1` (đó là dãy giảm).

**Ràng buộc:** số nút thuộc `[1, 3·10^4]`; `-3·10^4 <= Node.val <= 3·10^4`.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách level-order kiểu LeetCode (None = nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


examples = [
    ([1, None, 3, 2, 4, None, None, None, 5], 3),
    ([2, None, 3, 2, None, 1], 2),
]
for values, expected in examples:
    print(f"root={values} -> kỳ vọng {expected}")

root=[1, None, 3, 2, 4, None, None, None, 5] -> kỳ vọng 3
root=[2, None, 3, 2, None, 1] -> kỳ vọng 2


Cây của ví dụ 1 (đọc `build_tree`): gốc `1` có con phải `3`; `3` có con trái `2` và con phải `4`; `4` có con phải `5`.

```
    1
     \
      3
     / \
    2   4
         \
          5
```

## 3. Kiến thức nền

### 3.1. Đường đi xuống và độ dài tính theo số nút

Đường đi **xuống** đi từ một nút tới **con trực tiếp** rồi tiếp, không bao giờ lên cha. Độ dài tính theo **số nút** (một nút đơn lẻ có độ dài `1`), nên đáp án luôn `>= 1` vì `n >= 1`. Điều kiện "liên tiếp": nếu đang ở nút cha `p` và đi tới con `c` thì phải có `c.val == p.val + 1`.
#### Code demo

In [2]:
root = build_tree(examples[0][0])
print(root.val, root.right.val, root.right.left.val, root.right.right.val, root.right.right.right.val)   # 1 3 2 4 5
for name, (p, c) in {"1 -> 3": (root, root.right), "3 -> 2": (root.right, root.right.left), "3 -> 4": (root.right, root.right.right)}.items():
    print(f"{name}: con == cha + 1 ? {c.val == p.val + 1}")

1 3 2 4 5
1 -> 3: con == cha + 1 ? False
3 -> 2: con == cha + 1 ? False
3 -> 4: con == cha + 1 ? True


### 3.2. DFS và hai cách truyền thông tin

**DFS** duyệt sâu. Có hai cách để đồng bộ thông tin giữa cha và con:

- **Top-down (truyền xuống):** cha **gửi tham số** cho con (ví dụ "độ dài chuỗi hiện tại" và "giá trị của cha"). Con tự quyết định kéo dài hay bắt đầu lại.
- **Bottom-up (trả lên):** con **trả giá trị** cho cha (ví dụ "độ dài lớn nhất bắt đầu tại tôi"), cha tổng hợp.

Với bài này cả hai đều cho `O(n)`.
#### Code demo

In [3]:
def depth_top_down(node, d=1):
    if node is None:
        return 0
    return max(d, depth_top_down(node.left, d + 1), depth_top_down(node.right, d + 1))


def depth_bottom_up(node):
    if node is None:
        return 0
    return 1 + max(depth_bottom_up(node.left), depth_bottom_up(node.right))


print(depth_top_down(root), depth_bottom_up(root))         # 4 4 (cùng đáp án: chiều cao của cây)

4 4


### 3.3. Biến toàn cục `best` và `nonlocal`

Ở mỗi nút ta có thể phát hiện một chuỗi dài hơn kỷ lục cũ: dùng biến `best` ngoài hàm đệ quy và `nonlocal best` để cập nhật. Đừng quên cập nhật **ở mọi nút** (không chỉ ở lá).
#### Code demo

In [4]:
def count_nodes_nonlocal(node):
    total = 0

    def dfs(n):
        nonlocal total
        if n is None:
            return
        total += 1
        dfs(n.left)
        dfs(n.right)

    dfs(node)
    return total


print(count_nodes_nonlocal(root))     # 5

5


### 3.4. Hàng đợi / ngăn xếp để tránh đệ quy sâu

Python giới hạn độ sâu đệ quy (mặc định `1000`). Với cây lệch tới `3·10^4` nút ta cần duyệt bằng **stack** (`list`) hoặc **hàng đợi** (`deque`) tự quản, mỗi phần tử mang đủ thông tin (nút, độ dài hiện tại).
#### Code demo

In [5]:
stack = [(root, 1)]
order = []
while stack:
    node, d = stack.pop()
    order.append((node.val, d))
    for child in (node.right, node.left):
        if child:
            stack.append((child, d + 1))
print(order)                           # độ sâu tính từ gốc = 1

[(1, 1), (3, 2), (2, 3), (4, 3), (5, 4)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: đi từ gốc xuống và theo dõi độ dài chuỗi liên tiếp kết thúc tại mỗi nút:

- `1` (gốc): chuỗi `1`, độ dài `1`.
- `3` (con của `1`): `3 != 1 + 1` ⇒ chuỗi bị đứt, **bắt đầu lại**, độ dài `1`.
- `2` (con trái của `3`): `2 != 3 + 1` ⇒ độ dài `1`.
- `4` (con phải của `3`): `4 == 3 + 1` ⇒ độ dài `2`.
- `5` (con phải của `4`): `5 == 4 + 1` ⇒ độ dài `3`.

Lớn nhất `3`. **Cách nghĩ:** mỗi nút chỉ cần biết **độ dài chuỗi liên tiếp kết thúc ở cha nó** (và giá trị cha); nếu giá trị nó đúng bằng cha + 1 thì kéo dài thêm một, ngược lại bắt đầu lại từ `1`.

## 5. Cách 1 — Brute Force / baseline: với mỗi nút, DFS xuống đếm chuỗi bắt đầu từ nó

**Ý tưởng.** Với mỗi nút `v`, tính `from_node(v)` = độ dài dài nhất của dãy liên tiếp **bắt đầu tại `v`** đi xuống (thử cả hai con thỏa `child.val == v.val + 1`). Đáp án là max trên mọi nút.
#### Code demo

In [6]:
def longest_from(node):
    """Độ dài dãy liên tiếp dài nhất BẮT ĐẦU tại node (đi xuống)."""
    best = 1
    for child in (node.left, node.right):
        if child and child.val == node.val + 1:
            best = max(best, 1 + longest_from(child))
    return best


def longest_bruteforce(root):
    if root is None:
        return 0
    return max(longest_from(root), longest_bruteforce(root.left), longest_bruteforce(root.right))


for values, expected in examples:
    print(values, "->", longest_bruteforce(build_tree(values)), "(kỳ vọng", expected, ")")

[1, None, 3, 2, 4, None, None, None, 5] -> 3 (kỳ vọng 3 )
[2, None, 3, 2, None, 1] -> 2 (kỳ vọng 2 )


**Độ phức tạp (Cách 1).** Gọi `n` là số nút, `h` chiều cao. `longest_from(v)` thăm cây con của `v` (chỉ các nhánh liên tiếp); worst case (cả cây là một chuỗi `1,2,3,...,n`): tổng `n + (n-1) + ... = O(n²)`. **Time `O(n·h)` ⊆ `O(n²)`**, **Space `O(h)`**.

**Nút thắt.** Với chuỗi dài, ta **tính lại** cùng các đoạn nhiều lần: chuỗi bắt đầu ở nút `v` chứa chuỗi bắt đầu ở con của `v`. Một lượt duyệt top-down hoặc bottom-up là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

**Top-down:** mỗi nút nhận `(parent_val, length_ending_at_parent)`. Nếu `node.val == parent_val + 1` thì `length = length_ending_at_parent + 1`, ngược lại `length = 1`. Cập nhật `best = max(best, length)`, rồi gửi `(node.val, length)` xuống hai con. Mỗi nút làm `O(1)` việc ⇒ `O(n)`.

**Bottom-up:** `f(node)` = độ dài dài nhất bắt đầu tại `node`: `1 + max(f(child) nếu child.val == node.val + 1)`. Đáp án = max của `f` trên mọi nút (cập nhật khi tính xong mỗi nút).

## 7. Các cách tiếp theo

### Cách 2 — DFS từ trên xuống, mang `(giá trị cha, độ dài hiện tại)` (cách nộp chuẩn)

**Giải thích.** `dfs(node, parent_val, length_so_far)`: tính `length` cho `node`, cập nhật `best`, đệ quy cho hai con với `(node.val, length)`. Gọi `dfs(root, root.val - 2, 0)`: giá trị cha **giả** `root.val - 2` thỏa `root.val != cha + 1` nên gốc luôn bắt đầu chuỗi mới với độ dài `1`.
#### Code demo

In [7]:
def longest_top_down(root):
    best = 0

    def dfs(node, parent_val, length):
        nonlocal best
        if node is None:
            return
        length = length + 1 if node.val == parent_val + 1 else 1       # kéo dài hoặc bắt đầu lại
        best = max(best, length)                                        # cập nhật ở MỌI nút
        dfs(node.left, node.val, length)
        dfs(node.right, node.val, length)

    if root:
        dfs(root, root.val - 2, 0)                                      # giá trị cha giả: root.val != (root.val - 2) + 1 nên gốc bắt đầu chuỗi mới
    return best


for values, expected in examples:
    print(values, "->", longest_top_down(build_tree(values)), "(kỳ vọng", expected, ")")

[1, None, 3, 2, 4, None, None, None, 5] -> 3 (kỳ vọng 3 )
[2, None, 3, 2, None, 1] -> 2 (kỳ vọng 2 )


In [8]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        def dfs(root: Optional[TreeNode]) -> int:
            if root is None:
                return 0
            l = dfs(root.left) + 1
            r = dfs(root.right) + 1
            if root.left and root.left.val - root.val != 1:
                l = 1
            if root.right and root.right.val - root.val != 1:
                r = 1
            t = max(l, r)
            nonlocal ans
            ans = max(ans, t)
            return t

        ans = 0
        dfs(root)
        return ans


print(Solution().longestConsecutive(build_tree([1, None, 3, 2, 4, None, None, None, 5])))   # 3

3


(Bản nộp ở trên là dạng **bottom-up** gọn; Cách 2 ở cell trước là dạng top-down. Cả hai tương đương.)

**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(h)`** cho ngăn xếp đệ quy (với cây lệch `n = 3·10^4` vượt giới hạn đệ quy mặc định `1000` — xem Cách 4).

### Cách 3 — DFS từ dưới lên (trả về độ dài lớn nhất bắt đầu tại nút)

**Giải thích.** `f(node)` trả độ dài dài nhất của dãy bắt đầu tại `node`. Với con `c` thỏa `c.val == node.val + 1` thì ứng viên `1 + f(c)`. Cập nhật `best` ở mỗi nút. Trả `1` nếu không có con liên tiếp. (Gọi `f` cho **cả hai con** trước để không bỏ sót việc cập nhật `best` ở cây con không liên tiếp.)
#### Code demo

In [9]:
def longest_bottom_up(root):
    best = 0

    def f(node):
        nonlocal best
        if node is None:
            return 0
        left = f(node.left)                               # luôn gọi cả hai con (để duyệt hết cây)
        right = f(node.right)
        length = 1
        if node.left and node.left.val == node.val + 1:
            length = max(length, 1 + left)
        if node.right and node.right.val == node.val + 1:
            length = max(length, 1 + right)
        best = max(best, length)
        return length

    f(root)
    return best


for values, expected in examples:
    print(values, "->", longest_bottom_up(build_tree(values)), "(kỳ vọng", expected, ")")

[1, None, 3, 2, 4, None, None, None, 5] -> 3 (kỳ vọng 3 )
[2, None, 3, 2, None, 1] -> 2 (kỳ vọng 2 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(h)`**.

### Cách 4 — Lặp bằng stack (không đệ quy)

**Giải thích.** Mỗi phần tử stack là `(node, length)` với `length` là độ dài chuỗi kết thúc tại `node`. Lấy ra, cập nhật `best`, rồi đẩy hai con với độ dài mới (`length + 1` nếu `child.val == node.val + 1`, ngược lại `1`). Không dùng đệ quy nên chạy được với cây lệch `3·10^4` nút.
#### Code demo

In [10]:
def longest_iterative(root):
    if root is None:
        return 0
    best = 0
    stack = [(root, 1)]
    while stack:
        node, length = stack.pop()
        best = max(best, length)
        for child in (node.left, node.right):
            if child:
                stack.append((child, length + 1 if child.val == node.val + 1 else 1))
    return best


for values, expected in examples:
    print(values, "->", longest_iterative(build_tree(values)), "(kỳ vọng", expected, ")")

[1, None, 3, 2, 4, None, None, None, 5] -> 3 (kỳ vọng 3 )
[2, None, 3, 2, None, 1] -> 2 (kỳ vọng 2 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** worst (stack có thể chứa nhiều nút cùng lúc, ví dụ cây rộng) nhưng **không phụ thuộc giới hạn đệ quy**.

### Cách 5 — BFS theo từng nút (hàng đợi)

**Giải thích.** Giống Cách 4 nhưng dùng `deque` và duyệt theo tầng. Khác biệt chỉ là thứ tự duyệt; kết quả giống nhau.
#### Code demo

In [11]:
def longest_bfs(root):
    if root is None:
        return 0
    best = 0
    queue = deque([(root, 1)])
    while queue:
        node, length = queue.popleft()
        best = max(best, length)
        for child in (node.left, node.right):
            if child:
                queue.append((child, length + 1 if child.val == node.val + 1 else 1))
    return best


for values, expected in examples:
    print(values, "->", longest_bfs(build_tree(values)), "(kỳ vọng", expected, ")")

[1, None, 3, 2, 4, None, None, None, 5] -> 3 (kỳ vọng 3 )
[2, None, 3, 2, None, 1] -> 2 (kỳ vọng 2 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(w)`** với `w` là bề rộng lớn nhất của cây.

> Cách 2/3 là phổ biến nhất; Cách 4/5 an toàn cho cây rất sâu.

## 8. TRACE từng bước

Ta trace Cách 2 (top-down) trên ví dụ 1 và ví dụ 2, in `(giá trị cha, độ dài trước, độ dài mới)` ở mỗi nút.
#### Code demo

In [12]:
def trace_top_down(root):
    best = 0

    def dfs(node, parent_val, length, depth=0):
        nonlocal best
        if node is None:
            return
        extended = node.val == parent_val + 1
        new_length = length + 1 if extended else 1
        best = max(best, new_length)
        print(f"{'  ' * depth}nút {node.val}: cha={parent_val if depth else '-'}, "
              f"{'cha + 1 == nút -> kéo dài' if extended else 'không liên tiếp -> bắt đầu lại'}, độ dài {new_length}, best={best}")
        dfs(node.left, node.val, new_length, depth + 1)
        dfs(node.right, node.val, new_length, depth + 1)

    dfs(root, root.val - 2, 0)
    print("kết quả:", best)
    return best


trace_top_down(build_tree([1, None, 3, 2, 4, None, None, None, 5]))
print()
trace_top_down(build_tree([2, None, 3, 2, None, 1]))

nút 1: cha=-, không liên tiếp -> bắt đầu lại, độ dài 1, best=1
  nút 3: cha=1, không liên tiếp -> bắt đầu lại, độ dài 1, best=1
    nút 2: cha=3, không liên tiếp -> bắt đầu lại, độ dài 1, best=1
    nút 4: cha=3, cha + 1 == nút -> kéo dài, độ dài 2, best=2
      nút 5: cha=4, cha + 1 == nút -> kéo dài, độ dài 3, best=3
kết quả: 3

nút 2: cha=-, không liên tiếp -> bắt đầu lại, độ dài 1, best=1
  nút 3: cha=2, cha + 1 == nút -> kéo dài, độ dài 2, best=2
    nút 2: cha=3, không liên tiếp -> bắt đầu lại, độ dài 1, best=2
      nút 1: cha=2, không liên tiếp -> bắt đầu lại, độ dài 1, best=2
kết quả: 2


2

Bảng tóm tắt (ví dụ 1):

| nút | cha | liên tiếp? | độ dài kết thúc tại nút | `best` |
|---|---|---|---|---|
| `1` | — | — | 1 | 1 |
| `3` | `1` | `3 != 2` | 1 | 1 |
| `2` | `3` | `2 != 4` | 1 | 1 |
| `4` | `3` | `4 == 4` | 2 | 2 |
| `5` | `4` | `5 == 5` | 3 | 3 |

## 9. Code nộp LeetCode

Bản nộp dùng DFS (bottom-up gọn). Đúng `class Solution`, đúng `longestConsecutive(self, root: Optional[TreeNode]) -> int`.

In [13]:
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        def dfs(root: Optional[TreeNode]) -> int:
            if root is None:
                return 0
            l = dfs(root.left) + 1                              # chuỗi bắt đầu tại root đi sang trái
            r = dfs(root.right) + 1                             # ... sang phải
            if root.left and root.left.val - root.val != 1:     # con trái không đúng cha + 1: chuỗi đứt
                l = 1
            if root.right and root.right.val - root.val != 1:
                r = 1
            t = max(l, r)
            nonlocal ans
            ans = max(ans, t)                                   # cập nhật ở mọi nút
            return t

        ans = 0
        dfs(root)
        return ans


sol = Solution()
for values, expected in examples:
    assert sol.longestConsecutive(build_tree(values)) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Mô hình:** mọi đường đi liên tiếp là một đoạn **đứng yên** của một đường từ nút xuống con; nó **kết thúc** ở một nút `v`. Gọi `end(v)` là độ dài dài nhất của dãy liên tiếp **kết thúc tại `v`**: `end(v) = end(parent(v)) + 1` nếu `v.val == parent(v).val + 1`, ngược lại `1` (vì đường đi chỉ đi xuống nên nút đứng ngay trước `v` chỉ có thể là cha của nó; nhánh còn lại của cha không liên quan).
- **Top-down đúng:** truyền `end(parent)` và `parent.val` xuống để tính `end(v)` trong `O(1)`; đáp án `max_v end(v)`.
- **Bottom-up đúng (đối ngẫu):** gọi `start(v)` là độ dài dài nhất bắt đầu tại `v`: `start(v) = 1 + max{start(c) : c con của v, c.val == v.val + 1}` (hoặc `1`). Mỗi dãy liên tiếp cực đại bắt đầu ở một nút nào đó nên đáp án `max_v start(v)`.
- **Bản nộp:** `l = dfs(left) + 1` rồi đặt `l = 1` nếu con trái không liên tiếp; đúng `start(v)` cho nhánh trái (nếu không có con trái thì `dfs` trả `0` và `l = 1`). Dùng `+ 1` rồi ghi đè về `1` thay cho `if` rõ ràng.
- **Không đi lên:** mọi cách chỉ truyền thông tin từ cha xuống con hoặc từ con lên cha **đúng một chiều logic** của đường đi (từ trên xuống); ta không bao giờ nối hai nhánh con qua cha.
- **Mọi nút cập nhật `best`:** để chuỗi dài nhất có thể kết thúc/bắt đầu ở nút bất kỳ.

## 11. Phân tích Time / Space Complexity

Gọi `n` số nút, `h` chiều cao (`h` từ `log n` đến `n`).

| Cách | Time | Space |
|---|---|---|
| 1. Với mỗi nút DFS xuống | `O(n·h)` (worst `O(n²)`) | `O(h)` |
| 2. Top-down | `O(n)` | `O(h)` |
| 3. Bottom-up | `O(n)` | `O(h)` |
| 4. Lặp bằng stack | `O(n)` | `O(n)` worst |
| 5. BFS | `O(n)` | `O(w)` |

Với `n = 3·10^4`: Cách 2–5 cỡ `3·10^4` thao tác; Cách 1 có thể tới `~4.5·10^8` ở cây là một chuỗi `1,2,3,...`. Cây lệch sâu `3·10^4` làm Cách 2/3 chạm giới hạn đệ quy mặc định.

## 12. Bộ test

`oracle` là brute force độc lập: **với mỗi nút `v` làm điểm cuối**, đi ngược lên bằng **con trỏ cha** (tự gắn) khi `parent.val == cur.val - 1`, đếm số nút; đáp án là max trên mọi `v`. Cách này không dùng đệ quy xuống và không dùng `length + 1` xuôi. Sinh cây ngẫu nhiên có giá trị trong khoảng nhỏ để dãy liên tiếp hay xuất hiện.
#### Code demo

In [14]:
def all_nodes(root):
    out, stack = [], ([root] if root else [])
    while stack:
        node = stack.pop()
        out.append(node)
        for c in (node.left, node.right):
            if c:
                stack.append(c)
    return out


def oracle(root):
    parent = {}
    for node in all_nodes(root):
        for c in (node.left, node.right):
            if c:
                parent[c] = node
    best = 0
    for end in all_nodes(root):
        count, cur = 1, end
        while cur in parent and parent[cur].val == cur.val - 1:       # đi ngược lên khi giá trị giảm đúng 1
            cur = parent[cur]
            count += 1
        best = max(best, count)
    return best


def random_tree(n, lo, hi, rng, chain_bias=0.0):
    """Cây nhị phân ngẫu nhiên n nút, giá trị trong [lo, hi]; chain_bias > 0: ưu tiên giá trị = cha + 1."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        parent, side = slots.pop()
        value = parent.val + 1 if rng.random() < chain_bias else rng.randint(lo, hi)
        child = TreeNode(value)
        setattr(parent, side, child)
        slots += [(child, "left"), (child, "right")]
    return root


impls = {
    "bruteforce": longest_bruteforce,
    "top_down": longest_top_down,
    "bottom_up": longest_bottom_up,
    "iterative": longest_iterative,
    "bfs": longest_bfs,
    "submit": sol.longestConsecutive,
}

for values, expected in examples:
    root = build_tree(values)
    assert oracle(root) == expected
    for key, fn in impls.items():
        assert fn(root) == expected, (key, values, fn(root), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([1], 1, "một nút"),
    ([1, 2], 2, "con trái liên tiếp"),
    ([1, None, 2], 2, "con phải liên tiếp"),
    ([2, 1], 1, "giảm dần: không tính"),
    ([1, 2, 3], 2, "hai con: chỉ con trái 2 liên tiếp với gốc 1"),
    ([1, 2, None, 3, None, 4], 4, "chuỗi liên tiếp dọc nhánh trái"),
    ([1, 3, None, 4, None, 5], 3, "chuỗi bắt đầu giữa cây (3-4-5)"),
    ([5, 4, 3, 2, 1], 1, "toàn giảm dần"),
    ([1, 1, 1, 1], 1, "toàn bằng nhau"),
    ([-2, None, -1, None, 0, None, 1], 4, "giá trị âm"),
    ([30000, None, -30000], 1, "giá trị biên của đề"),
    ([1, 2, 2, 3, 3, 3, 3], 3, "nhiều nhánh liên tiếp cùng độ dài"),
]
for values, expected, label in edge_cases:
    root = build_tree(values)
    assert oracle(root) == expected, (label, oracle(root), expected)
    for key, fn in impls.items():
        assert fn(root) == expected, (key, label, fn(root), expected)
    print("PASS", label)

PASS một nút
PASS con trái liên tiếp
PASS con phải liên tiếp
PASS giảm dần: không tính
PASS hai con: chỉ con trái 2 liên tiếp với gốc 1
PASS chuỗi liên tiếp dọc nhánh trái
PASS chuỗi bắt đầu giữa cây (3-4-5)
PASS toàn giảm dần
PASS toàn bằng nhau
PASS giá trị âm
PASS giá trị biên của đề
PASS nhiều nhánh liên tiếp cùng độ dài


### 12.2. Tính chất và phản ví dụ

- Đáp án `>= 1` và `<=` chiều cao của cây (tính theo nút).
- **Cộng cùng một hằng** vào mọi giá trị không đổi đáp án; **đổi dấu** `x → -x` biến chuỗi tăng thành giảm nên **không** bất biến (kiểm tra ngược lại).
- **Phản ví dụ** cho lỗi cho phép giảm dần: ví dụ 2 `[2,null,3,2,null,1]` có đáp án `2`, nhưng nếu cho phép cả bước giảm thì đường `2-3-2-1` (mỗi bước chênh `1`) được tính và ra `4`.
#### Code demo

In [16]:
rng = random.Random(298)
for _ in range(400):
    n = rng.randint(1, 40)
    root = random_tree(n, -5, 5, rng, chain_bias=rng.choice([0.0, 0.5, 0.9]))
    ans = sol.longestConsecutive(root)
    height = depth_bottom_up(root)
    assert 1 <= ans <= height
    d = rng.randint(-100, 100)
    for node in all_nodes(root):
        node.val += d
    assert sol.longestConsecutive(root) == ans
print("PASS tính chất (cận, cộng hằng số) trên 400 cây ngẫu nhiên")


def sai_cho_phep_giam(root):
    best = 0

    def dfs(node, parent_val, length):
        nonlocal best
        if node is None:
            return
        length = length + 1 if abs(node.val - parent_val) == 1 else 1       # SAI: cho cả giảm dần
        best = max(best, length)
        dfs(node.left, node.val, length)
        dfs(node.right, node.val, length)

    if root:
        dfs(root, root.val - 10, 0)
    return best


bad = build_tree([2, None, 3, 2, None, 1])
print("phản ví dụ ví dụ 2: đúng", sol.longestConsecutive(bad), "| cho phép giảm:", sai_cho_phep_giam(bad))
assert sai_cho_phep_giam(bad) > sol.longestConsecutive(bad)

PASS tính chất (cận, cộng hằng số) trên 400 cây ngẫu nhiên
phản ví dụ ví dụ 2: đúng 2 | cho phép giảm: 4


### 12.3. Đối chiếu ngẫu nhiên (có seed) và cây lớn/sâu
#### Code demo

In [17]:
rng = random.Random(4298)
for trial in range(2000):
    n = rng.randint(1, 60)
    hi = rng.choice([2, 4, 10, 30])
    root = random_tree(n, 0, hi, rng, chain_bias=rng.choice([0.0, 0.3, 0.7, 0.95]))
    truth = oracle(root)
    for key, fn in impls.items():
        assert fn(root) == truth, (key, n, fn(root), truth)
print("PASS 2000 cây ngẫu nhiên (nhiều chuỗi liên tiếp, nhiều ngắt quãng): mọi cách khớp oracle")

# Cây lệch sâu 30000 nút: chuỗi 1, 2, 3, ..., n (đáp án n) và chuỗi bị ngắt giữa chừng
def chain_tree(values):
    root = TreeNode(values[0])
    cur = root
    for v in values[1:]:
        cur.right = TreeNode(v)
        cur = cur.right
    return root


n = 30_000
inc = chain_tree(list(range(1, n + 1)))
broken = chain_tree(list(range(1, 15_001)) + list(range(100, 100 + 15_000)))
for name, tree, expected in [("tăng dần 30000", inc, n), ("ngắt giữa", broken, 15_000)]:
    for key in ("iterative", "bfs"):
        t0 = time.perf_counter()
        assert impls[key](tree) == expected, (name, key)
        print(f"{name:<16} {key:<9} {(time.perf_counter() - t0) * 1000:7.1f} ms")
try:
    impls["top_down"](inc)
    print("top_down trên cây sâu: không lỗi")
except RecursionError:
    print("top_down trên cây sâu 30000: RecursionError (giới hạn đệ quy mặc định) -> dùng bản lặp")
print("PASS cây sâu 30000 nút")

PASS 2000 cây ngẫu nhiên (nhiều chuỗi liên tiếp, nhiều ngắt quãng): mọi cách khớp oracle


tăng dần 30000   iterative     5.4 ms
tăng dần 30000   bfs           4.8 ms
ngắt giữa        iterative     5.4 ms
ngắt giữa        bfs           4.7 ms
top_down trên cây sâu 30000: RecursionError (giới hạn đệ quy mặc định) -> dùng bản lặp
PASS cây sâu 30000 nút


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cho phép chuỗi giảm dần (hoặc theo cả hai chiều)
Xem `sai_cho_phep_giam` ở phần test: đường `2-3-2-1` bị tính (cho đáp án `4` thay vì `2`). **Vì sao sai:** đề yêu cầu **tăng dần đúng `1`** dọc theo hướng đi xuống. **Code đúng:** `child.val == parent.val + 1`.

### Lỗi 2: Không reset độ dài về `1` khi chuỗi đứt
#### Code sai

In [18]:
def sai_khong_reset(root):
    best = 0

    def dfs(node, parent_val, length):
        nonlocal best
        if node is None:
            return
        if node.val == parent_val + 1:
            length += 1
        # SAI: thiếu else length = 1; độ dài cứ cộng dồn qua cả chỗ đứt
        best = max(best, length)
        dfs(node.left, node.val, length)
        dfs(node.right, node.val, length)

    if root:
        dfs(root, root.val - 2, 1)
    return best


for values in ([1, None, 3, 2, 4, None, None, None, 5], [1, 2, None, 5, None, 6]):
    t = build_tree(values)
    print(values, "đúng:", sol.longestConsecutive(t), "| sai:", sai_khong_reset(t))

[1, None, 3, 2, 4, None, None, None, 5] đúng: 3 | sai: 3
[1, 2, None, 5, None, 6] đúng: 2 | sai: 3


**Vì sao sai:** khi `node.val != parent_val + 1` chuỗi mới bắt đầu, độ dài phải về `1`. Nếu giữ nguyên độ dài cũ thì nối nhầm hai chuỗi rời nhau.

### Lỗi 3: So sánh với giá trị của gốc thay vì của cha
#### Code sai

In [19]:
def sai_so_voi_goc(root):
    best = 0

    def dfs(node, root_val, depth):
        nonlocal best
        if node is None:
            return
        if node.val == root_val + depth:                  # SAI: so với gốc, không phải với cha trực tiếp
            best = max(best, depth + 1)
        dfs(node.left, root_val, depth + 1)
        dfs(node.right, root_val, depth + 1)

    if root:
        dfs(root, root.val, 0)
    return max(best, 1)


for values in ([1, None, 3, 2, 4, None, None, None, 5], [4, None, 5, 6]):
    t = build_tree(values)
    print(values, "đúng:", sol.longestConsecutive(t), "| sai:", sai_so_voi_goc(t))

[1, None, 3, 2, 4, None, None, None, 5] đúng: 3 | sai: 1
[4, None, 5, 6] đúng: 3 | sai: 3


**Vì sao sai:** chuỗi có thể bắt đầu ở **nút bất kỳ**, không chỉ ở gốc (ví dụ `3-4-5` bắt đầu ở nút `3`). Phải so mỗi nút với **cha trực tiếp** của nó.

### Lỗi 4: Không cập nhật `best` ở mọi nút (chỉ ở lá)
#### Code sai

In [20]:
def sai_chi_cap_nhat_o_la(root):
    best = 0

    def dfs(node, parent_val, length):
        nonlocal best
        if node is None:
            return
        length = length + 1 if node.val == parent_val + 1 else 1
        if node.left is None and node.right is None:               # SAI: chỉ cập nhật khi tới lá
            best = max(best, length)
        dfs(node.left, node.val, length)
        dfs(node.right, node.val, length)

    if root:
        dfs(root, root.val - 2, 0)
    return best


bad = build_tree([1, 2, 3, 4, 5])
print("[1,2,3,4,5]: đúng", sol.longestConsecutive(bad), "| chỉ ở lá:", sai_chi_cap_nhat_o_la(bad))
t = build_tree([1, 2, None, 3, None, 10])
print([1, 2, None, 3, None, 10], "đúng:", sol.longestConsecutive(t), "| chỉ ở lá:", sai_chi_cap_nhat_o_la(t))

[1,2,3,4,5]: đúng 2 | chỉ ở lá: 1
[1, 2, None, 3, None, 10] đúng: 3 | chỉ ở lá: 1


**Vì sao sai:** chuỗi dài nhất có thể **kết thúc ở nút trong** (có con không liên tiếp với nó), nên không được bỏ qua các nút không phải lá. **Code đúng:** `best = max(best, length)` ở mọi nút.

### Lỗi 5: Đếm cạnh thay vì nút (hoặc quên `+1`)
Đề yêu cầu **số nút** (độ dài đường đi theo nút). Một nút đơn lẻ có độ dài `1` (không phải `0`). Nếu bạn đếm cạnh, mọi đáp án thiếu `1`.
#### Code sai

In [21]:
def sai_dem_canh(root):
    return longest_iterative(root) - 1                            # SAI: đếm cạnh


print("đúng:", sol.longestConsecutive(build_tree([1, None, 3, 2, 4, None, None, None, 5])), "| đếm cạnh:", sai_dem_canh(build_tree([1, None, 3, 2, 4, None, None, None, 5])))

đúng: 3 | đếm cạnh: 2


**Vì sao sai:** `3-4-5` có `3` nút và `2` cạnh; đề hỏi `3`.

### Lỗi 6: Đệ quy trên cây lệch sâu
Xem phần 12.3: Cách 2/3 đệ quy chạm `RecursionError` ở chuỗi `30000` nút. **Cách sửa:** bản lặp (Cách 4/5) hoặc nâng `sys.setrecursionlimit`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**đường đi xuống thỏa điều kiện giữa cha và con**, bắt đầu/kết thúc bất kỳ" → duyệt một lượt, mỗi nút **kế thừa** `(giá trị cha, độ dài)` từ cha: kéo dài nếu thỏa, ngược lại **reset về 1**.
- Phân biệt: đường **từ trên xuống** (bài này) và đường **bất kỳ qua một nút** (Binary Tree Longest Consecutive Sequence II — LeetCode 549 — mới cần ghép hai nhánh).
- Cập nhật kỷ lục **ở mọi nút**.
- Cây có thể lệch sâu `~3·10^4`: biết cách viết bản **lặp** (stack/queue).
- Top-down (tham số) và bottom-up (giá trị trả về) là hai cách nghĩ tương đương — chọn cách dễ viết nhất.
- Họ hàng: **Longest Univalue Path** (687), **Diameter of Binary Tree** (543), **Path Sum** (112), **Binary Tree Maximum Path Sum** (124).

## 15. Biến thể của bài

1. **Longest Consecutive Sequence II** (LeetCode 549): dãy liên tiếp có thể **tăng hoặc giảm** và có thể **đi qua một nút cha** (nối hai nhánh).
2. **Longest Univalue Path** (687): các nút liên tiếp có **cùng giá trị**.
3. **Đếm số đường đi** liên tiếp độ dài `>= k`.
4. **In ra chính đường đi** dài nhất (không chỉ độ dài).
5. Cây **tổng quát** (nhiều con): lặp qua danh sách con.
#### Code demo

In [22]:
def longest_path_nodes(root):
    """Trả đường đi (danh sách giá trị) dài nhất theo bài này."""
    best_path = []
    stack = [(root, [root.val])]
    while stack:
        node, path = stack.pop()
        if len(path) > len(best_path):
            best_path = path
        for child in (node.left, node.right):
            if child:
                stack.append((child, path + [child.val] if child.val == node.val + 1 else [child.val]))
    return best_path


def count_paths_at_least(root, k):
    count = 0
    stack = [(root, 1)]
    while stack:
        node, length = stack.pop()
        if length >= k:
            count += 1                      # đếm các nút mà chuỗi kết thúc tại đó có độ dài >= k
        for child in (node.left, node.right):
            if child:
                stack.append((child, length + 1 if child.val == node.val + 1 else 1))
    return count


t = build_tree([1, None, 3, 2, 4, None, None, None, 5])
print(longest_path_nodes(t), count_paths_at_least(t, 2))        # [3, 4, 5] 2

[3, 4, 5] 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS xuống từ mỗi nút | `longest_from(v)` cho từng `v` | `O(n·h)` | `O(h)` | baseline |
| 2. Top-down | mang `(giá trị cha, độ dài)` xuống | `O(n)` | `O(h)` | tư duy "kế thừa từ cha" |
| 3. Bottom-up | con trả "dài nhất bắt đầu tại nó" | `O(n)` | `O(h)` | **bản nộp gọn** |
| 4. Stack lặp | `(nút, độ dài)` trong stack | `O(n)` | `O(n)` | cây rất sâu |
| 5. BFS | `(nút, độ dài)` trong hàng đợi | `O(n)` | `O(w)` | cây rất sâu, theo tầng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().longestConsecutive(build_tree([1]))`; (b) `build_tree([1, 2, 3])` có đáp án bao nhiêu; (c) `build_tree([3, 2, 1])`; (d) `build_tree([1, None, 2, None, 3, None, 4])`.

**Bài 2 (tính bằng tay).** Với cây `[10, 11, 12, 12, 13, 13, 14]`, ghi độ dài chuỗi kết thúc tại mỗi nút theo Cách 2 và đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `longest_todo(root)` theo Cách 4 (lặp).
#### Code demo

In [23]:
def longest_todo(root):
    if root is None:
        return 0
    best = 0
    # TODO 1: stack = [(root, 1)]
    # TODO 2: while stack: node, length = stack.pop(); best = max(best, length)
    # TODO 3: với mỗi con (trái, phải) không rỗng: độ dài mới = length + 1 nếu child.val == node.val + 1, ngược lại 1
    # TODO 4: đẩy (child, độ dài mới) vào stack
    # TODO 5: trả best
    pass


# Tự làm xong rồi mới mở test:
# for values, expected in examples:
#     assert longest_todo(build_tree(values)) == expected

**Bài 4 (tự viết lại).** Viết `longest_consecutive_ii(root)` (LeetCode 549) cho phép dãy **tăng hoặc giảm** (không cần đi qua nút cha) bằng cách trả về cặp `(dài nhất tăng bắt đầu tại nút, dài nhất giảm bắt đầu tại nút)`; đối chiếu với brute force.

**Bài 5 (biến thể / mini challenge).** Viết `longest_univalue_path_down(root)`: đường đi **xuống** dài nhất (theo nút) mà mọi nút cùng giá trị; thử `[5,5,5,5,5,None,5]` (đáp án `3`: gốc → con trái → cháu trái).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: chỉ tính khi con = cha + 1. Bài 2: đi từ gốc xuống; nhánh `12` hai nút. Bài 5: đổi điều kiện thành `child.val == node.val`.
- **Mức 2 (vừa):** Bài 1: `1`, `2`, `1`, `4`. Bài 2: tự trace theo Cách 2 rồi đối chiếu với kết quả `Solution().longestConsecutive` ở phần đáp án. Bài 3: khung Cách 4.
- **Mức 3 (sâu):** Bài 4: mỗi nút trả `(inc, dec)`; `inc = 1 + max(inc(c))` nếu `c.val == v.val + 1`; `dec` tương tự với `-1`; đáp án là `max(inc, dec)` (đường chỉ đi xuống, một chiều). Bài 5: `[5,5,5,5,5,None,5]` có cây con trái `5-5-5` ⇒ `3`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert Solution().longestConsecutive(build_tree([1])) == 1
assert Solution().longestConsecutive(build_tree([1, 2, 3])) == 2
assert Solution().longestConsecutive(build_tree([3, 2, 1])) == 1
assert Solution().longestConsecutive(build_tree([1, None, 2, None, 3, None, 4])) == 4

# Bài 2
t10 = build_tree([10, 11, 12, 12, 13, 13, 14])
assert oracle(t10) == Solution().longestConsecutive(t10)
print("[10,11,12,12,13,13,14] ->", Solution().longestConsecutive(t10))


# Bài 3
def longest_todo_ans(root):
    if root is None:
        return 0
    best = 0
    stack = [(root, 1)]
    while stack:
        node, length = stack.pop()
        best = max(best, length)
        for child in (node.left, node.right):
            if child:
                stack.append((child, length + 1 if child.val == node.val + 1 else 1))
    return best


for values, expected in examples:
    assert longest_todo_ans(build_tree(values)) == expected


# Bài 4 (dạng đi xuống, tăng hoặc giảm)
def longest_consecutive_ii_down(root):
    best = 0

    def dfs(node):
        nonlocal best
        if node is None:
            return 0, 0
        inc = dec = 1
        for child in (node.left, node.right):
            if child:
                ci, cd = dfs(child)
                if child.val == node.val + 1:
                    inc = max(inc, 1 + ci)
                elif child.val == node.val - 1:
                    dec = max(dec, 1 + cd)
        best = max(best, inc, dec)
        return inc, dec

    dfs(root)
    return best


assert longest_consecutive_ii_down(build_tree([2, None, 3, 2, None, 1])) == 3
rng = random.Random(19)
for _ in range(200):
    tree = random_tree(rng.randint(1, 30), 0, 6, rng, chain_bias=0.5)
    # brute: dãy liên tiếp tăng hoặc giảm đi xuống
    def brute(root):
        best = 1
        for start in all_nodes(root):
            for step in (1, -1):
                stack = [(start, 1)]
                while stack:
                    node, length = stack.pop()
                    best = max(best, length)
                    for child in (node.left, node.right):
                        if child and child.val == node.val + step:
                            stack.append((child, length + 1))
        return best
    assert longest_consecutive_ii_down(tree) == brute(tree)


# Bài 5
def longest_univalue_path_down(root):
    best = 0
    stack = [(root, 1)]
    while stack:
        node, length = stack.pop()
        best = max(best, length)
        for child in (node.left, node.right):
            if child:
                stack.append((child, length + 1 if child.val == node.val else 1))
    return best


assert longest_univalue_path_down(build_tree([5, 5, 5, 5, 5, None, 5])) == 3
print("PASS đáp án bài tập")

[10,11,12,12,13,13,14] -> 3
PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DFS bottom-up gọn)

```python
class Solution:
    def longestConsecutive(self, root: Optional[TreeNode]) -> int:
        def dfs(root):
            if root is None:
                return 0
            l = dfs(root.left) + 1
            r = dfs(root.right) + 1
            if root.left and root.left.val - root.val != 1:
                l = 1
            if root.right and root.right.val - root.val != 1:
                r = 1
            t = max(l, r)
            nonlocal ans
            ans = max(ans, t)
            return t

        ans = 0
        dfs(root)
        return ans
```

**Checklist**

- [ ] Điều kiện liên tiếp: `child.val == parent.val + 1` (tăng, so với **cha trực tiếp**)?
- [ ] Chuỗi đứt ⇒ **reset về `1`**?
- [ ] Cập nhật `best` ở **mọi nút**; độ dài tính theo **nút** (một nút = `1`)?
- [ ] Chỉ đi **xuống** (không nối hai nhánh qua cha)?
- [ ] Cây lệch sâu (`3·10^4`) ⇒ biết bản **lặp** hoặc nâng giới hạn đệ quy?
- [ ] Đã thử một nút, toàn giảm dần, chuỗi bắt đầu giữa cây, toàn bằng nhau, giá trị âm?
- [ ] Đã đối chiếu với brute force "đi ngược lên cha" trên cây ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def longestConsecutive(self, root: Optional[TreeNode]) -> int`?